In [22]:
import os
from torchvision import datasets , transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Load Data

In [4]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness = 0.2 , contrast = 0.2),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean =[0.485 , 0.456 , 0.406] , std =[0.229 , 0.224 , 0.225])  # from imagenet
])

In [5]:
dataset_path = "./dataset"

dataset = datasets.ImageFolder(dataset_path , transform = image_transforms)
len(dataset)

2300

In [6]:
dataset.classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [7]:
num_classes = len(dataset.classes)
num_classes

6

In [8]:
train_size = int(0.75 * len(dataset))
val_size = len(dataset) - train_size
train_size , val_size

(1725, 575)

In [9]:
from torch.utils.data import random_split

train_dataset , val_dataset = random_split(dataset , [train_size , val_size])

In [10]:
train_loader = DataLoader(train_dataset , batch_size=32 , shuffle = True)
val_loader = DataLoader(val_dataset , batch_size=32 , shuffle=True)

In [11]:
def train_model(model , criterion , optimizer , epochs = 5):
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num , (images , labels) in enumerate(train_loader):
            images , labels = images.to(device) , labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            #forward pass
            outputs = model(images)
            loss = criterion(outputs , labels)
            
            #backword pass and optimizer
            loss.backward()
            optimizer.step()

            if (batch_num+1) % 10 == 0:
                print(f"Batch : {batch_num+1} , Epoch: {epoch+1} , Loss: {loss.item(): 0.2f}")

            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] , Avg loss: {epoch_loss: .4f}")

        # validation
        model.eval()
        correct=0
        total=0
        all_labels = []
        all_predictions =[]

        with torch.no_grad():
            for images , labels in val_loader:
                images , labels = images.to(device) , labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data,1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())
            print(f"*** Validation Accuracy : {100 * correct / total:.2f}% ***")
    end = time.time()
    print(f"Execution time : {end-start} seconds")
    return all_labels , all_predictions      

In [12]:
class CarClassifierEfficientNetf3(nn.Module):
    def __init__(self , num_classes):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')

        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze last feature block
        for param in self.model.features[-2].parameters():
            param.requires_grad = True

        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features , num_classes)
        )

    def forward(self , x):
        x = self.model(x)
        return x 

        

In [13]:
model = CarClassifierEfficientNetf3(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p:p.requires_grad , model.parameters()) , lr =0.001)
 
labels , predictions = train_model(model , criterion , optimizer , epochs =10)

Batch : 10 , Epoch: 1 , Loss:  1.44
Batch : 20 , Epoch: 1 , Loss:  1.19
Batch : 30 , Epoch: 1 , Loss:  1.02
Batch : 40 , Epoch: 1 , Loss:  0.79
Batch : 50 , Epoch: 1 , Loss:  1.17
Epoch [1/10] , Avg loss:  1.1112
*** Validation Accuracy : 70.26% ***
Batch : 10 , Epoch: 2 , Loss:  0.67
Batch : 20 , Epoch: 2 , Loss:  0.48
Batch : 30 , Epoch: 2 , Loss:  0.64
Batch : 40 , Epoch: 2 , Loss:  0.73
Batch : 50 , Epoch: 2 , Loss:  0.80
Epoch [2/10] , Avg loss:  0.6531
*** Validation Accuracy : 74.26% ***
Batch : 10 , Epoch: 3 , Loss:  0.38
Batch : 20 , Epoch: 3 , Loss:  0.63
Batch : 30 , Epoch: 3 , Loss:  0.44
Batch : 40 , Epoch: 3 , Loss:  0.48
Batch : 50 , Epoch: 3 , Loss:  0.56
Epoch [3/10] , Avg loss:  0.5073
*** Validation Accuracy : 74.96% ***
Batch : 10 , Epoch: 4 , Loss:  0.25
Batch : 20 , Epoch: 4 , Loss:  0.27
Batch : 30 , Epoch: 4 , Loss:  0.46
Batch : 40 , Epoch: 4 , Loss:  0.43
Batch : 50 , Epoch: 4 , Loss:  0.36
Epoch [4/10] , Avg loss:  0.4288
*** Validation Accuracy : 77.22% ***


### Hyperparameter tuning

In [19]:
class CarClassifierEfficientNetf3(nn.Module):
    def __init__(self , num_classes , dropout_rate = 0.5):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')

        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze last feature block
        for param in self.model.features[-2].parameters():
            param.requires_grad = True

        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features , num_classes)
        )

    def forward(self , x):
        x = self.model(x)
        return x 

        

In [20]:
# define objective function for optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr' , 1e-5 , 1e-2 , log=True)
    dropout_rate = trial.suggest_float('dropout_rate' , 0.2 , 0.7)

    # load the model
    model = CarClassifierEfficientNetf3(num_classes = num_classes , dropout_rate=dropout_rate).to(device)

    #difine the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad , model.parameters()), lr = lr)

    # Training loop
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num ,(images , labels) in enumerate(train_loader):
            images, labels = images.to(device) , labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs , labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images , labels in val_loader:
                images , labels = images.to(device) , labels.to(device)
                outputs = model(images)
                _,predicted = torch.max(outputs.data , 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct/total
        
        # report imidiate results to optuna
        trial.report(accuracy , epoch)

        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time : {int((end-start)/60)} minutes and {(end-start)%60} seconds ")

    return accuracy
        
                

In [23]:
#create the study and optimize
study = optuna.create_study(direction ='maximize') 
study.optimize(objective , n_trials=20)

[I 2025-12-24 00:45:28,773] A new study created in memory with name: no-name-1da3b8b8-9b86-4c24-9df7-ddffe9c08ca6
[I 2025-12-24 00:56:09,532] Trial 0 finished with value: 68.34782608695652 and parameters: {'lr': 0.00016497945462878787, 'dropout_rate': 0.3762573363777504}. Best is trial 0 with value: 68.34782608695652.


Execution time : 10 minutes and 40.575618743896484 seconds 


[I 2025-12-24 01:06:33,946] Trial 1 finished with value: 72.69565217391305 and parameters: {'lr': 0.004482440012420653, 'dropout_rate': 0.6498020349681155}. Best is trial 1 with value: 72.69565217391305.


Execution time : 10 minutes and 24.217708349227905 seconds 


[I 2025-12-24 01:16:21,073] Trial 2 finished with value: 39.130434782608695 and parameters: {'lr': 1.9516464212540357e-05, 'dropout_rate': 0.6066512305324443}. Best is trial 1 with value: 72.69565217391305.


...

[I 2025-12-24 03:31:12,786] Trial 16 finished with value: 76.17391304347827 and parameters: {'lr': 0.0007898955687664665, 'dropout_rate': 0.3644067527086873}. Best is trial 16 with value: 76.17391304347827.


Execution time : 9 minutes and 14.764584064483643 seconds 


[I 2025-12-24 03:40:32,768] Trial 17 finished with value: 74.08695652173913 and parameters: {'lr': 0.0006950264328146312, 'dropout_rate': 0.35783599107934044}. Best is trial 16 with value: 76.17391304347827.


Execution time : 9 minutes and 19.68428921699524 seconds 


[I 2025-12-24 03:43:38,367] Trial 18 pruned. 
[I 2025-12-24 03:46:42,778] Trial 19 pruned. 


In [24]:
best_params = study.best_params
print(best_params)

{'lr': 0.0007898955687664665, 'dropout_rate': 0.3644067527086873}


In [25]:
class CarClassifierEfficientNetf3(nn.Module):
    def __init__(self , num_classes):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')

        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze last feature block
        for param in self.model.features[-2].parameters():
            param.requires_grad = True

        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(0.36),
            nn.Linear(in_features , num_classes)
        )

    def forward(self , x):
        x = self.model(x)
        return x 

        

In [26]:
model = CarClassifierEfficientNetf3(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p:p.requires_grad , model.parameters()) , lr =0.0008)
 
labels , predictions = train_model(model , criterion , optimizer , epochs =10)

Batch : 10 , Epoch: 1 , Loss:  1.39
Batch : 20 , Epoch: 1 , Loss:  1.05
Batch : 30 , Epoch: 1 , Loss:  0.97
Batch : 40 , Epoch: 1 , Loss:  1.24
Batch : 50 , Epoch: 1 , Loss:  0.92
Epoch [1/10] , Avg loss:  1.1195
*** Validation Accuracy : 69.91% ***
Batch : 10 , Epoch: 2 , Loss:  0.61
Batch : 20 , Epoch: 2 , Loss:  0.72
Batch : 30 , Epoch: 2 , Loss:  0.54
Batch : 40 , Epoch: 2 , Loss:  0.57
Batch : 50 , Epoch: 2 , Loss:  0.65
Epoch [2/10] , Avg loss:  0.6702
*** Validation Accuracy : 73.57% ***
Batch : 10 , Epoch: 3 , Loss:  0.53
Batch : 20 , Epoch: 3 , Loss:  0.46
Batch : 30 , Epoch: 3 , Loss:  0.47
Batch : 40 , Epoch: 3 , Loss:  0.55
Batch : 50 , Epoch: 3 , Loss:  0.61
Epoch [3/10] , Avg loss:  0.5395
*** Validation Accuracy : 76.17% ***
Batch : 10 , Epoch: 4 , Loss:  0.52
Batch : 20 , Epoch: 4 , Loss:  0.34
Batch : 30 , Epoch: 4 , Loss:  0.35
Batch : 40 , Epoch: 4 , Loss:  0.45
Batch : 50 , Epoch: 4 , Loss:  0.43
Epoch [4/10] , Avg loss:  0.4549
*** Validation Accuracy : 74.09% ***
